# Kaggle runner: smoke test, then (later) the reproduction run

Before running: **Accelerator = GPU (T4)**, **Internet = On**, and add the Kaggle dataset
`ealtman2019/ibm-transactions-for-anti-money-laundering-aml` as an input.

Every step stops the notebook if it fails, so later cells never run on a broken state.
This file is written by `python -m tools.make_notebooks`; change the generator, not this file.

Safe by default: `RUN_FULL = False` runs a smoke test:
1. stock Multi-GNN run, 3 epochs, scored and recorded (`..._smoke_stock`)
2. wrapper runs: source, then 10% scratch, then 10% fine-tune (`..._smoke`), each capped at 2 epochs.
   Smoke runs always keep a last-epoch checkpoint.

Set `RUN_FULL = True` only after the owner has approved the GPU time (AGENTS.md rule 6).

In [ ]:
import os, shlex, subprocess, sys, time


def run(command):
    """Run a shell command, show its output as it arrives, and stop the notebook if it fails."""
    process = subprocess.Popen(
        ["bash", "-o", "pipefail", "-c", command],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if code != 0:
        raise RuntimeError(f"Command failed (exit code {code}): {command}")


REPO_URL = "https://github.com/coachuttam/aml-gnn-transfer.git"

KAGGLE_DATA = "/kaggle/input/ibm-transactions-for-anti-money-laundering-aml"
RUN_FULL = False  # False = smoke test only; True = full reproduction run
WORK = "/kaggle/working"
REPO = f"{WORK}/repo"

os.environ["AML_DATA_ROOT"] = f"{WORK}/data"
os.environ["AML_MODEL_ROOT"] = f"{WORK}/models"
# Keep __pycache__ out of the Multi-GNN submodule folder.
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

# Stop here if the GPU accelerator is off; remember which GPU we got.
run("nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv")
GPU_NAME = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
).splitlines()[0].strip()
print("GPU:", GPU_NAME)

In [ ]:
# Always run the latest pushed code: update an existing checkout instead of keeping a stale one.
if os.path.isdir(REPO):
    run(f"git -C {REPO} fetch origin && git -C {REPO} reset --hard origin/main")
else:
    run(f"git clone {REPO_URL} {REPO}")
run(f"git -C {REPO} submodule update --init")
run(f"git -C {REPO} log -1 --oneline")
run(f"git -C {REPO} submodule status")
os.chdir(REPO)
sys.path.insert(0, REPO)

In [ ]:
from src.config import (
    MULTI_GNN_ADAPTATION_FLAGS, REPRODUCTION_EPOCHS, SEEDS, SMOKE_LABEL_FRACTION,
    SMOKE_STEP_BUDGET_EPOCHS, SMOKE_TEST_EPOCHS,
)
from src.records import make_run_id, smoke_run_id

# The stock run gets its own id in smoke mode, so it never collides with the wrapper's.
REAL_RUN_ID = make_run_id("gin", "source", 1.0, SEEDS[0])
STOCK_RUN_ID = REAL_RUN_ID if RUN_FULL else smoke_run_id(REAL_RUN_ID) + "_stock"
print("Stock run id:", STOCK_RUN_ID)

Environment. First choice is Multi-GNN's own `env.yml` (micromamba, forced to a CUDA build of torch),
plus pyg-lib (the fast neighbour sampler, which `env.yml` lacks) as a wheel matched to that torch.
Whatever we get must pass `src.check_environment`: CUDA works, the PyG extensions import and run,
pyg-lib is in use. If it does not, we fall back to the notebook's own CUDA torch with all PyG add-ons
installed as matching wheels (`src.pyg_wheels`). If that also fails the check, the notebook stops.

In [ ]:
MICROMAMBA = f"{WORK}/bin/micromamba"
ENV_PYTHON = f"{MICROMAMBA} run -r {WORK}/mamba -n multignn python"

try:
    run(f"curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C {WORK} bin/micromamba")
    run(f"{MICROMAMBA} create -y -r {WORK}/mamba -n multignn -f external/Multi-GNN/env.yml 'pytorch=*=*cuda11.8*'")
    run(f"{ENV_PYTHON} -m src.pyg_wheels --only pyg_lib")
    run(f"{ENV_PYTHON} -m src.check_environment")
    PYTHON, ENV_LABEL = ENV_PYTHON, "env.yml"
except RuntimeError as problem:
    print("\nThe env.yml environment is not usable:", problem)
    print("Falling back to this notebook's torch with matching PyG wheels.\n")
    run(f"{sys.executable} -m src.pyg_wheels")
    run(f"{sys.executable} -m src.check_environment")  # raises if still wrong
    PYTHON, ENV_LABEL = sys.executable, "pip-fallback"
ENVIRONMENT_LABEL = f"{ENV_LABEL} / {GPU_NAME}"
print("Using:", ENVIRONMENT_LABEL)

Format HI-Small and LI-Small with IBM's script and write the stock run's `data_config.json` (CPU only, slow).

In [ ]:
run(f"{PYTHON} -m src.prepare --raw-dir {KAGGLE_DATA} --run-id {STOCK_RUN_ID}")

Stock Multi-GIN + EU on HI-Small (Multi-GNN's own `main.py`, unmodified).

In [ ]:
EPOCHS = REPRODUCTION_EPOCHS if RUN_FULL else SMOKE_TEST_EPOCHS
FLAGS = " ".join(MULTI_GNN_ADAPTATION_FLAGS)
MAIN = f"{REPO}/external/Multi-GNN/main.py"

# Start from the run folder so Multi-GNN reads OUR data_config.json.
os.chdir(f"{WORK}/data/multignn_runs/{STOCK_RUN_ID}")
start = time.time()
run(f"{PYTHON} {MAIN} --data Small_HI --model gin {FLAGS} --seed {SEEDS[0]} --n_epochs {EPOCHS} --testing --save_model --unique_name")
RUNTIME_SECONDS = time.time() - start
os.chdir(REPO)
print(f"{EPOCHS} epochs took {RUNTIME_SECONDS / 60:.1f} min in total (includes data loading)")

Score the stock run's checkpoint (threshold on validation, test scored once) and write its record.
Records land in `repo/results/runs/`; download them from the notebook output and commit them.

In [ ]:
smoke_flag = "" if RUN_FULL else "--smoke-test"
run(f"{PYTHON} -m src.record_stock_run --run-id {STOCK_RUN_ID} --model gin --dataset HI-Small --seed {SEEDS[0]} --epochs {EPOCHS} --runtime-seconds {RUNTIME_SECONDS:.0f} --environment {shlex.quote(ENVIRONMENT_LABEL)} {smoke_flag}")
run(f"cat results/runs/{STOCK_RUN_ID}.json")

Wrapper smoke test (skipped when `RUN_FULL` is on): source, then 10% scratch, then 10% fine-tune
from that source. Each run is capped at 2 epochs, validates on the same schedule as real runs, and
writes a `_smoke` record. The chain stops at the first failure.

In [ ]:
def run_timed(label, command):
    """Run one step and print how long it took and on which GPU."""
    start = time.time()
    run(command)
    print(f"[{label}] {(time.time() - start) / 60:.1f} min on {GPU_NAME}", flush=True)


def smoke_chain(python, environment_label, seed, budget, fraction):
    """Source, then scratch and finetune at `fraction`, each as a smoke run."""
    label = shlex.quote(environment_label)
    for mode in ("source", "scratch", "finetune"):
        share = 1.0 if mode == "source" else fraction
        print(f"\n===== wrapper smoke run: {mode}, fraction {share} =====")
        run_timed(
            f"smoke {mode}",
            f"{python} -m src.run_experiment --model gin --mode {mode} --fraction {share} "
            f"--seed {seed} --step-budget-epochs {budget} --environment {label} --smoke-test",
        )

if RUN_FULL:
    print("RUN_FULL is on: skipping the wrapper smoke runs")
else:
    smoke_chain(PYTHON, ENVIRONMENT_LABEL, SEEDS[0], SMOKE_STEP_BUDGET_EPOCHS, SMOKE_LABEL_FRACTION)
    run(f"{PYTHON} -m src.check_smoke")

Per-epoch scores of the stock run. Pick the best **validation** epoch and read test from that epoch.

In [ ]:
run(f"grep -E 'Validation F1|Test F1' {WORK}/data/multignn_runs/{STOCK_RUN_ID}/logs/logs.log || true")